# Distillation: ConvNeXt-Tiny taught by the CNN + RAD-DINO average (NIH official split, seed 42)

**Question.** Can a ConvNeXt-Tiny trained on the soft predictions of the CNN + RAD-DINO average reach RAD-DINO-level accuracy at the CNN's inference cost (0.18x RAD-DINO at batch 32)? RAD-DINO is used at training time only.

**Rules:** `evaluation_protocol.yaml` v2.2.0, section `distillation`, written before any distillation result. The cascade notebook (`2-staged_official_split.ipynb`) is not changed.

| Session | Flag | Hardware | Time (T4) | Output |
|---|---|---|---|---|
| D1 | `RUN_TEACHER` | GPU | ~45 min | teacher QC (D1-D3), `soft_labels_train.npz`, `distill_go_no_go.json` |
| D2 | `RUN_STUDENT` | GPU | ~3 h | run folder `..._M10_CONVNEXT_TINY_KD_384__s42` (checkpoint, val/test logits), QC-D4 |
| D3 | `RUN_EVAL` | CPU | ~15 min | T10-T11, `distill_outcome.json` (helps / matches / beats RAD-DINO) |

D1 and D2 can run in one GPU session: D2 starts only if the go/no-go rule says **go**.

**Inputs to attach on Kaggle**
1. `nih-chest-xrays/data` (images, `Data_Entry_2017.csv`, official list files).
2. A dataset with `results_s42/` (at least `calibration.json` and the two `runs/` folders with their logits), and each teacher's `checkpoint_best.pt` placed inside its run folder:
   - `NIH_CXR14__NIH_OFFICIAL_PV__M08_CONVNEXT_TINY_ASL_384_CR__be576650cff5__s42__r0/checkpoint_best.pt` (sha256 `eebd5f66...`)
   - `NIH_CXR14__NIH_OFFICIAL_PV__M09_RAD_DINO_518_CR__cb7d09842184__s42__r0/checkpoint_best.pt` (sha256 `070f90c7...`, **not** `checkpoint_last.pt`)
3. Internet on (ImageNet weights for the student, RAD-DINO config at the pinned revision).

In [ ]:
# ============================ CONFIG ============================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")      # determinism; set before CUDA starts
import json, glob, time, math, random, hashlib, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import rankdata
warnings.filterwarnings("ignore")

# ---- what to run in THIS session ----
RUN_TEACHER  = True     # Session D1 (GPU, ~45 min): rebuild both teachers, QC, soft labels on train, go/no-go
RUN_STUDENT  = True     # Session D2 (GPU, ~3 h): train M10 on 0.5*y + 0.5*q - runs only if go/no-go says "go"
RUN_EVAL     = False    # Session D3 (CPU, ~15 min): calibration, thresholds, test metrics, bootstrap, outcome
DEBUG        = False    # small subset + 1+1 student epochs; folders get a _debug tag and must never be reported

SEED = 42               # the only training seed of this study

PROTOCOL_VERSION = "2.2.0"
OUT_DIR   = os.environ.get("CXR_OUT_DIR", "/kaggle/working")
INPUT_DIR = os.environ.get("CXR_INPUT_DIR", "/kaggle/input")
DATA_ROOT = "/kaggle/input/datasets/nih-chest-xrays/data"      # NIH images + Data_Entry_2017.csv (sessions A/B)
NUM_WORKERS = 4

DATASET_ID, SPLIT_ID = "NIH_CXR14", "NIH_OFFICIAL_PV"
VAL_FRACTION, SPLIT_SEED = 0.15, 42                              # validation = 15 % of train_val patients
EXPECTED_COUNTS = {"train_val": 86524, "test": 25596, "total": 112120}
EXPECTED_LIST_SHA256 = {                                          # SHA-256 of sorted, LF-joined image ids
    "train_val_list.txt": "61fbe896321c1c1c8b75f3e4f3a08e4fef6486d95ef8a667c31d4d60dca6cb81",
    "test_list.txt":      "38ca5ef7f756092946f57c1a59faca882ed589a1ab1f72590b45dc06c6d5e1cc"}
EXPECTED_SPLIT_SHA256 = "b9f2fbc9aa061dafa76994494b92779651dc3ddd2ed3e7b82c70bbf5cf7071dd"   # split of the seed-42 runs

CLASSES = ["Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule", "Pneumonia",
           "Pneumothorax", "Consolidation", "Edema", "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"]
C = len(CLASSES)

ASL = dict(gamma_neg=4, gamma_pos=1, clip=0.05)
CNN_CFG = dict(
    kind="cnn", model_id="M08_CONVNEXT_TINY_ASL_384_CR", timm_name="convnext_tiny.fb_in22k_ft_in1k",
    img_size=384, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225),
    batch_size=32, warmup_epochs=1, finetune_epochs=8, patience=2,
    lr_head_warmup=1e-3, lr_head=1e-4, lr_backbone=1e-4,
    weight_decay=0.05, drop_path=0.1, use_ema=False, ema_decay=0.999, grad_clip_norm=1.0, asl=ASL, time_budget_h=8.0)
VIT_CFG = dict(
    kind="raddino", model_id="M09_RAD_DINO_518_CR", hf_name="microsoft/rad-dino", hf_revision="110cbc18d5133582e320b43d53bf5c44e410c936",
    img_size=518, mean=(0.5307,) * 3, std=(0.2583,) * 3,
    batch_size=12, warmup_epochs=2, finetune_epochs=4, patience=2, unfreeze_last=8, layer_decay=0.8,
    lr_head_warmup=1e-3, lr_head=2e-4, lr_backbone=2e-5,
    weight_decay=0.05, drop_path=0.1, use_ema=True, ema_decay=0.999, grad_clip_norm=1.0, asl=ASL, time_budget_h=9.0)

# ---- distillation (protocol section distillation, fixed before any distillation result) ----
STUDENT_CFG = dict(CNN_CFG, model_id="M10_CONVNEXT_TINY_KD_384")   # M08 recipe; only the training target changes
KD_LAMBDA = 0.5                  # target t = (1 - KD_LAMBDA) * y + KD_LAMBDA * q
GO_MAX_TRAIN_AUROC = 0.95        # go only if the teacher's train macro AUROC is below this
TEACHER_MEMBERS = {              # model_id -> (config, sha256 of checkpoint_best.pt)
    "M08_CONVNEXT_TINY_ASL_384_CR": (CNN_CFG, "eebd5f662d397d2fedeb5aa3fc70a2b59d70b05923e6c6712bf91049b9df70ad"),
    "M09_RAD_DINO_518_CR":          (VIT_CFG, "070f90c7cf44c54447a015791f5a18d0eb0d8b05bce070450b6c35e91008bd93")}
CAL_KEYS = {"M08_CONVNEXT_TINY_ASL_384_CR": "cnn", "M09_RAD_DINO_518_CR": "vit"}   # keys in v2.0.0 calibration.json

# ---- evaluation (session D3; same settings as v2.0.0) ----
N_BOOT, BOOTSTRAP_SEED, ALPHA = 1000, 20260922, 0.05              # patient-level bootstrap
ECE_BINS = 15
MIN_SUPPORT_WARN = 100
MATCH_MARGIN = 0.005             # "matches RAD-DINO": lower CI bound of Distilled - RAD-DINO macro AUROC > -0.005
V2_LATENCY_MS = {                # v2.0.0 session-T medians (Tesla T4); used if the timing files are not attached
    "M08_CONVNEXT_TINY_ASL_384_CR": {1: 11.371520042419434, 32: 4.0095114707946777},
    "M09_RAD_DINO_518_CR":          {1: 22.170719146728516, 32: 22.849385261535645}}

if DEBUG:                        # teachers keep their trained input size; only the student is shrunk
    STUDENT_CFG.update(img_size=224, warmup_epochs=1, finetune_epochs=1, batch_size=16)
    DATASET_ID += "_debug"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- helpers ----------------
def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def write_json(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)

def save_table(df_, name):
    """CSV, UTF-8, full precision (rounding only when displayed)."""
    df_.to_csv(f"{OUT_DIR}/{name}.csv", index=False, float_format="%.17g", na_rep="")

def fmt(x, d=4):
    return "n/a" if x is None or not np.isfinite(x) else f"{np.round(x, d):.{d}f}"

def _walk(bases=(OUT_DIR, INPUT_DIR)):
    """os.walk over outputs and inputs, skipping the 112k-image folders."""
    for base in dict.fromkeys(bases):
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if not d.startswith("images")]
            yield root, dirs, files

def find_file(fname):
    for root, _, files in _walk():
        if fname in files:
            return os.path.join(root, fname)
    return None

def run_folder_name(model_id):
    return f"{DATASET_ID}__{SPLIT_ID}__{model_id}__s{SEED}"

def find_run_dir(model_id):
    """Seed-42 run folder of a model that contains exported logits. Accepts the older names that carry a
    config hash (…__{model_id}__{hash}__s42__r0). More than one distinct folder is an error."""
    hits = {}
    for root, dirs, files in _walk():
        name = os.path.basename(root)
        if ("logits_test.npz" in files and name.startswith(f"{DATASET_ID}__{SPLIT_ID}__{model_id}__")
                and (name.endswith(f"__s{SEED}") or f"__s{SEED}__" in name) and "probe" not in name):
            hits.setdefault(name, root)                    # OUT_DIR is searched first
    assert len(hits) <= 1, f"several seed-{SEED} folders for {model_id}: {sorted(hits)} - attach only one"
    return next(iter(hits.values()), None)

## 1. Data and official split (identical code to the cascade notebook, so the split hash reproduces)

In [ ]:
NEED_DATA = RUN_TEACHER or RUN_STUDENT
if NEED_DATA:
    from sklearn.model_selection import GroupShuffleSplit
    meta_csv = os.path.join(DATA_ROOT, "Data_Entry_2017.csv")
    if not os.path.exists(meta_csv):
        meta_csv = next((os.path.join(r, f) for r, _, fs in os.walk(INPUT_DIR) for f in fs if f == "Data_Entry_2017.csv"), meta_csv)
    assert os.path.exists(meta_csv), "Data_Entry_2017.csv not found - check DATA_ROOT"
    DATA_ROOT = os.path.dirname(meta_csv)
    png = glob.glob(os.path.join(DATA_ROOT, "images_*", "images", "*.png")) or \
          [os.path.join(r, f) for r, _, fs in os.walk(DATA_ROOT) for f in fs if f.endswith(".png")]
    df = pd.read_csv(meta_csv)
    df["path"] = df["Image Index"].map({os.path.basename(p): p for p in png})
    assert len(df) == EXPECTED_COUNTS["total"] and df.path.notna().all(), "images missing on disk"
    labels = df["Finding Labels"].str.split("|")
    for c in CLASSES:
        df[c] = labels.apply(lambda L, c=c: c in L).astype(np.float32)
    df = df.sort_values("Image Index").reset_index(drop=True)      # canonical order

    # ---- official list files (QC-01) ----
    def read_list(fname):
        p = os.path.join(DATA_ROOT, fname)
        p = p if os.path.exists(p) else find_file(fname)
        assert p, f"{fname} not found"
        return open(p).read().split()
    tv_ids, te_ids = read_list("train_val_list.txt"), read_list("test_list.txt")
    list_sha = {"train_val_list.txt": sha256_bytes("\n".join(sorted(tv_ids)).encode()),
                "test_list.txt": sha256_bytes("\n".join(sorted(te_ids)).encode())}
    assert (len(tv_ids), len(te_ids)) == (EXPECTED_COUNTS["train_val"], EXPECTED_COUNTS["test"]), "QC-01: list counts"
    assert list_sha == EXPECTED_LIST_SHA256, "QC-01: list files differ from the official lists"
    assert not set(tv_ids) & set(te_ids) and set(tv_ids) | set(te_ids) == set(df["Image Index"]), "QC-01: overlap/coverage"

    # ---- split (identical code to the one that produced the seed-42 runs) ----
    df["split"] = np.where(df["Image Index"].isin(set(te_ids)), "test", "trainval")
    tv_frame = df[df.split == "trainval"]
    _, va = next(GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SPLIT_SEED)
                 .split(tv_frame, groups=tv_frame["Patient ID"]))
    df.loc[tv_frame.index, "split"] = "train"
    df.loc[tv_frame.index[va], "split"] = "val"
    split_text = (df[["Image Index", "Patient ID", "split"]]
                  .rename(columns={"Image Index": "image_index", "Patient ID": "patient_id", "split": "partition"})
                  .sort_values("image_index").to_csv(index=False, lineterminator="\n"))
    SPLIT_SHA256 = sha256_bytes(split_text.encode("utf-8"))
    assert DEBUG or SPLIT_SHA256 == EXPECTED_SPLIT_SHA256, \
        f"split hash {SPLIT_SHA256[:12]} != {EXPECTED_SPLIT_SHA256[:12]}: library versions changed the validation carve-out"
    with open(f"{OUT_DIR}/split_{SPLIT_ID}.csv", "w", encoding="utf-8", newline="\n") as f:
        f.write(split_text)
    pats = {s: set(df.loc[df.split == s, "Patient ID"]) for s in ("train", "val", "test")}
    assert not (pats["train"] & pats["val"] or pats["train"] & pats["test"] or pats["val"] & pats["test"]), "QC-02: patient leakage"

    df_train, df_val, df_test = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
    if DEBUG:
        df_train, df_val, df_test = (d.sample(n, random_state=SPLIT_SEED).reset_index(drop=True)
                                     for d, n in ((df_train, 3000), (df_val, 1000), (df_test, 1000)))

    # ---- T1 dataset characteristics ----
    t1 = []
    for part, fr in (("train", df_train), ("val", df_val), ("test", df_test)):
        Y = fr[CLASSES].values
        base = dict(dataset_id=DATASET_ID, split_id=SPLIT_ID, partition=part, n_images=len(fr),
                    n_patients=fr["Patient ID"].nunique(), label_cardinality_mean=Y.sum(1).mean(),
                    n_all_negative_images=int((Y.sum(1) == 0).sum()),
                    images_per_patient_mean=len(fr) / fr["Patient ID"].nunique(),
                    view_PA_fraction=(fr["View Position"] == "PA").mean(), male_fraction=(fr["Patient Gender"] == "M").mean(),
                    age_median=fr["Patient Age"].median(), age_q25=fr["Patient Age"].quantile(.25),
                    age_q75=fr["Patient Age"].quantile(.75))
        t1.append(dict(base, label="ALL", n_positive=int(Y.sum()), prevalence=np.nan))
        t1 += [dict(base, label=c, n_positive=int(Y[:, i].sum()), prevalence=Y[:, i].mean()) for i, c in enumerate(CLASSES)]
    save_table(pd.DataFrame(t1), "T1_dataset_characteristics")
    print(f"split {SPLIT_ID} ({SPLIT_SHA256[:12]}) | train {len(df_train):,} | val {len(df_val):,} | test {len(df_test):,}")

## 2. Model and training code (cascade notebook code; changes: pinned RAD-DINO revision, optional training targets)

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

def set_determinism():
    torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
    torch.use_deterministic_algorithms(True, warn_only=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); random.seed(s)

def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

class CXRDataset(Dataset):
    def __init__(self, paths, y, cfg, train):
        self.paths, self.y, s = paths, y.astype(np.float32), cfg["img_size"]
        self.tfm = (T.Compose([T.RandomResizedCrop(s, scale=(0.85, 1.0), ratio=(0.9, 1.1)), T.RandomHorizontalFlip(0.5),
                               T.RandomApply([T.RandomRotation(7)], p=0.5), T.ColorJitter(brightness=0.1, contrast=0.1),
                               T.ToTensor()]) if train else
                    T.Compose([T.Resize((s, s), interpolation=T.InterpolationMode.BILINEAR), T.ToTensor()]))
        self.norm = T.Normalize(cfg["mean"], cfg["std"])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("L")               # grayscale -> [0, 1] -> 3 channels
        return self.norm(self.tfm(img).repeat(3, 1, 1)), self.y[i]

def loader(frame, cfg, train, batch_size=None, y=None):
    """y: training targets in frame order (soft targets for the student); default = the hard labels."""
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(CXRDataset(frame["path"].values, frame[CLASSES].values if y is None else y, cfg, train),
                      batch_size=batch_size or cfg["batch_size"], shuffle=train, num_workers=NUM_WORKERS,
                      pin_memory=USE_AMP, persistent_workers=NUM_WORKERS > 0, drop_last=False,
                      worker_init_fn=worker_init_fn, generator=g)

class AsymmetricLoss(nn.Module):
    """ASL (Ridnik et al., 2021)."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits.float())
        p_neg = (p - self.clip).clamp(min=0) if self.clip > 0 else p
        loss_pos = y * torch.log(p.clamp(min=self.eps)) * (1 - p).pow(self.gp)
        loss_neg = (1 - y) * torch.log((1 - p_neg).clamp(min=self.eps)) * p_neg.pow(self.gn)
        return -(loss_pos + loss_neg).sum(1).mean()

class CNNClassifier(nn.Module):
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        import timm
        self.backbone = timm.create_model(cfg["timm_name"], pretrained=pretrained, num_classes=0, drop_path_rate=cfg["drop_path"])
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(self.backbone.num_features, C))

    def forward(self, x):
        return self.head(self.backbone(x))

    def backbone_param_groups(self, cfg):
        return [{"params": list(self.backbone.parameters()), "lr": cfg["lr_backbone"]}]

class RadDinoClassifier(nn.Module):
    """RAD-DINO ViT-B/14 + dual pooling (CLS ⊕ mean patch = 1536-d)."""
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        from transformers import AutoModel, AutoConfig
        self.backbone = (AutoModel.from_pretrained(cfg["hf_name"], revision=cfg.get("hf_revision"), drop_path_rate=cfg["drop_path"]) if pretrained else
                         AutoModel.from_config(AutoConfig.from_pretrained(cfg["hf_name"], revision=cfg.get("hf_revision"), drop_path_rate=cfg["drop_path"])))
        self.n_reg = getattr(self.backbone.config, "num_register_tokens", 0)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(2 * self.backbone.config.hidden_size, C))

    def forward(self, x):
        h = self.backbone(pixel_values=x).last_hidden_state
        return self.head(torch.cat([h[:, 0], h[:, 1 + self.n_reg:].mean(1)], dim=1))

    def backbone_param_groups(self, cfg):
        layers, k = self.backbone.encoder.layer, cfg["unfreeze_last"]
        n = len(layers)
        groups = [{"params": list(self.backbone.layernorm.parameters()), "lr": cfg["lr_backbone"]}]
        groups += [{"params": list(layers[i].parameters()), "lr": cfg["lr_backbone"] * cfg["layer_decay"] ** (n - 1 - i)}
                   for i in range(n - k, n)]
        return groups

def build_model(cfg, pretrained=True):
    seed_everything(SEED)                                          # seeded head initialisation
    return (CNNClassifier(cfg, pretrained) if cfg["kind"] == "cnn" else RadDinoClassifier(cfg, pretrained)).to(DEVICE)

@torch.no_grad()
def predict_logits(model, dl):
    model.eval()
    out = []
    for x, _ in dl:
        with autocast():
            out.append(model(x.to(DEVICE, non_blocking=True)).float().cpu())
    return torch.cat(out).numpy().astype(np.float32)

def configure_phase(model, cfg, phase):
    """warmup = head only; finetune = head + (part of) the backbone."""
    for p in model.parameters():
        p.requires_grad = False
    groups = [{"params": list(model.head.parameters()), "lr": cfg["lr_head_warmup"] if phase == "warmup" else cfg["lr_head"]}]
    if phase == "finetune":
        groups += model.backbone_param_groups(cfg)
    for g in groups:
        for p in g["params"]:
            p.requires_grad = True
    return torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])

def cosine_schedule(opt, total_steps, warmup_frac=0.03):
    w = max(1, int(total_steps * warmup_frac))
    f = lambda s: (s + 1) / w if s < w else 0.5 * (1 + math.cos(math.pi * (s - w) / max(1, total_steps - w)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def val_macro_auroc(y, s):
    from sklearn.metrics import roc_auc_score
    return float(np.mean([roc_auc_score(y[:, c], s[:, c]) for c in range(C) if 0 < y[:, c].sum() < len(y)]))

def train_model(cfg, exp_dir, model, y_train=None):
    loss_fn = AsymmetricLoss(**cfg["asl"])
    train_dl, val_dl = loader(df_train, cfg, True, y=y_train), loader(df_val, cfg, False, cfg["batch_size"] * 2)
    yv = df_val[CLASSES].values
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    plan = [("warmup", i) for i in range(cfg["warmup_epochs"])] + [("finetune", i) for i in range(cfg["finetune_epochs"])]
    best, best_epoch, bad, ema, phase_now, t0, ep_times, truncated = -1.0, 0, 0, None, None, time.time(), [], False
    for ep, (phase, _) in enumerate(plan, 1):
        if ep_times and (time.time() - t0 + max(ep_times)) / 3600 > cfg["time_budget_h"]:
            print("Stopping: next epoch would exceed the time budget -> truncated=true"); truncated = True
            break
        if phase != phase_now:
            phase_now, bad = phase, 0
            opt = configure_phase(model, cfg, phase)
            n_ep = cfg["warmup_epochs"] if phase == "warmup" else cfg["finetune_epochs"]
            sched = cosine_schedule(opt, n_ep * len(train_dl))
            if phase == "finetune" and cfg["use_ema"]:
                from timm.utils import ModelEmaV2
                ema = ModelEmaV2(model, decay=cfg["ema_decay"])
        if USE_AMP:
            torch.cuda.reset_peak_memory_stats()
        model.train()
        te, run_loss = time.time(), 0.0
        for i, (x, y) in enumerate(train_dl):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with autocast():
                logits = model(x)
            loss = loss_fn(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg["grad_clip_norm"])
            scaler.step(opt); scaler.update(); sched.step()
            if ema is not None:
                ema.update(model)
            run_loss += loss.item()
            if i % 500 == 0:
                print(f"  epoch {ep} step {i}/{len(train_dl)} loss {loss.item():.4f}", flush=True)
        eval_model = ema.module if ema is not None else model                 # EMA weights are evaluated (M09)
        zv = predict_logits(eval_model, val_dl)
        auc = val_macro_auroc(yv, zv)
        ep_times.append(time.time() - te)
        row = dict(epoch=ep, phase=phase, train_loss=run_loss / len(train_dl),
                   val_loss=float(loss_fn(torch.as_tensor(zv), torch.as_tensor(yv))), val_macro_auroc=auc,
                   epoch_seconds=ep_times[-1], peak_gpu_mem_mb=torch.cuda.max_memory_allocated() / 2 ** 20 if USE_AMP else np.nan)
        log = f"{exp_dir}/train_log.csv"
        pd.DataFrame([row]).to_csv(log, mode="a", header=not os.path.exists(log), index=False)
        print(f"Epoch {ep} [{phase}] loss {row['train_loss']:.4f} | val macro AUROC {auc:.4f} | {ep_times[-1] / 60:.1f} min", flush=True)
        if auc > best:                                                        # strict '>' keeps the earlier epoch on ties
            best, best_epoch, bad = auc, ep, 0
            torch.save(eval_model.state_dict(), f"{exp_dir}/checkpoint_best.pt")
        elif phase == "finetune":
            bad += 1
            if bad >= cfg["patience"]:
                print("Early stopping."); break
    model.load_state_dict(torch.load(f"{exp_dir}/checkpoint_best.pt", map_location=DEVICE))
    return model, dict(best_val_macro_auroc=best, best_epoch=best_epoch, epochs_trained=len(ep_times),
                       truncated=truncated, train_wall_clock_h=(time.time() - t0) / 3600)

def run_stage(cfg, y_train=None, extra_info=None):
    """Train one model with SEED and export val/test logits. Skips if the seed-42 logits already exist.
    y_train: training targets in df_train order (default: hard labels); extra_info: stored in config.json."""
    existing = find_run_dir(cfg["model_id"])
    if existing:
        print(f"{cfg['model_id']}: logits already at {existing} - skipping."); return
    exp_dir = f"{OUT_DIR}/{run_folder_name(cfg['model_id'])}"
    os.makedirs(exp_dir, exist_ok=True)
    set_determinism()
    model = build_model(cfg)
    model, info = train_model(cfg, exp_dir, model, y_train)
    for part, frame in (("val", df_val), ("test", df_test)):
        z = predict_logits(model, loader(frame, cfg, False, cfg["batch_size"] * 2))
        np.savez(f"{exp_dir}/logits_{part}.npz", image_index=frame["Image Index"].to_numpy(dtype=str),
                 patient_id=frame["Patient ID"].to_numpy(dtype=np.int64), logits=z,
                 labels=frame[CLASSES].to_numpy(dtype=np.uint8), label_order=json.dumps(CLASSES),
                 split_id=SPLIT_ID, split_sha256=SPLIT_SHA256, model_id=cfg["model_id"], seed=SEED, eval_set=part)
    import timm, transformers
    write_json(f"{exp_dir}/config.json", dict(
        model_id=cfg["model_id"], seed=SEED, split_id=SPLIT_ID, split_sha256=SPLIT_SHA256, protocol_version=PROTOCOL_VERSION,
        model_cfg=cfg, train_info=info, checkpoint_sha256=sha256_file(f"{exp_dir}/checkpoint_best.pt"),
        software=dict(torch=torch.__version__, timm=timm.__version__, transformers=transformers.__version__),
        gpu=torch.cuda.get_device_name(0) if USE_AMP else "cpu", **(extra_info or {})))
    print(f"{cfg['model_id']}: done ({info['train_wall_clock_h']:.2f} h, best epoch {info['best_epoch']})")


## 3. Session D1: rebuild the teachers, check them, write soft labels, apply the go/no-go rule

QC-D1: checkpoint hashes. QC-D2: each rebuilt teacher reproduces its saved validation logits. QC-D3: soft labels only for train images. Go if the teacher's train macro AUROC is < 0.95.

In [ ]:
# ===================== Session D1: teacher soft labels + go/no-go (GPU) =====================
TEACHER_DIR = f"{OUT_DIR}/teacher"
QC_D = {}

def find_checkpoint(model_id, sha):
    """checkpoint_best.pt in a seed-42 run folder of model_id whose sha256 equals the recorded hash (QC-D1)."""
    seen = []
    for root, _, files in _walk():
        if "checkpoint_best.pt" in files and os.path.basename(root).startswith(f"{DATASET_ID.replace('_debug', '')}__{SPLIT_ID}__{model_id}__"):
            p = os.path.join(root, "checkpoint_best.pt")
            h = sha256_file(p)
            if h == sha:
                return p
            seen.append((p, h[:12]))
    raise FileNotFoundError(f"QC-D1: no checkpoint_best.pt for {model_id} with sha256 {sha[:12]}... - found {seen}")

def platt(z, model_id):
    c = CAL[CAL_KEYS[model_id]]
    return expit(np.asarray(c["a"]) * z + np.asarray(c["b"]))

def ids(frame):
    return frame["Image Index"].to_numpy(dtype=str)

if RUN_TEACHER:
    assert USE_AMP, "session D1 needs a GPU"
    cal_path = find_file("calibration.json")
    assert cal_path, "calibration.json (v2.0.0, results_s42) not found - attach it as input"
    CAL = json.load(open(cal_path))
    os.makedirs(TEACHER_DIR, exist_ok=True)
    p_train, p_val = {}, {}
    for model_id, (tcfg, sha) in TEACHER_MEMBERS.items():
        t0 = time.time()
        model = build_model(tcfg, pretrained=False)
        model.load_state_dict(torch.load(find_checkpoint(model_id, sha), map_location=DEVICE))
        model.eval()
        QC_D[f"QC-D1 {model_id}"] = f"pass ({sha[:12]})"
        bs = tcfg["batch_size"] * 2

        # QC-D2: the rebuilt teacher must reproduce its saved v2.0.0 validation logits
        zv = predict_logits(model, loader(df_val, tcfg, False, bs))
        np.savez(f"{TEACHER_DIR}/{model_id}_logits_val_rebuilt.npz", image_index=ids(df_val), logits=zv)
        if DEBUG:
            QC_D[f"QC-D2 {model_id}"] = "skipped (DEBUG subset)"
        else:
            saved = np.load(os.path.join(find_run_dir(model_id), "logits_val.npz"))
            assert (saved["image_index"] == ids(df_val)).all(), f"QC-D2 {model_id}: validation image order differs"
            yv = df_val[CLASSES].values
            corr = min(np.corrcoef(zv[:, c], saved["logits"][:, c])[0, 1] for c in range(C))
            d_auc = abs(val_macro_auroc(yv, zv) - val_macro_auroc(yv, saved["logits"]))
            ok = corr >= 0.999 and d_auc <= 0.001
            QC_D[f"QC-D2 {model_id}"] = f"{'pass' if ok else 'FAIL'} (min per-class corr {corr:.5f}, |d macro AUROC| {d_auc:.5f})"
            assert ok, QC_D[f"QC-D2 {model_id}"]

        zt = predict_logits(model, loader(df_train, tcfg, False, bs))
        np.savez(f"{TEACHER_DIR}/{model_id}_logits_train.npz", image_index=ids(df_train),
                 patient_id=df_train["Patient ID"].to_numpy(dtype=np.int64), logits=zt,
                 labels=df_train[CLASSES].to_numpy(dtype=np.uint8), label_order=json.dumps(CLASSES),
                 checkpoint_sha256=sha, split_id=SPLIT_ID, split_sha256=SPLIT_SHA256)
        p_train[model_id], p_val[model_id] = platt(zt, model_id), platt(zv, model_id)
        print(f"{model_id}: {QC_D[f'QC-D2 {model_id}']} | train + val pass {(time.time() - t0) / 60:.1f} min", flush=True)
        del model
        torch.cuda.empty_cache()

    # soft labels: train images only, df_train order (QC-D3)
    q = np.mean([p_train[m] for m in TEACHER_MEMBERS], axis=0).astype(np.float32)
    leak = set(ids(df_train)) & (set(ids(df_val)) | set(ids(df_test)))
    QC_D["QC-D3"] = "pass" if len(q) == len(df_train) and not leak else f"FAIL ({len(leak)} val/test images)"
    assert QC_D["QC-D3"] == "pass", QC_D["QC-D3"]
    np.savez(f"{OUT_DIR}/soft_labels_train.npz", image_index=ids(df_train), q=q,
             labels=df_train[CLASSES].to_numpy(dtype=np.uint8), label_order=json.dumps(CLASSES),
             kd_lambda=KD_LAMBDA, split_sha256=SPLIT_SHA256)

    # go / no-go (protocol distillation.go_no_go)
    yt, yv = df_train[CLASSES].values, df_val[CLASSES].values
    auc = {f"{m} train": val_macro_auroc(yt, p_train[m]) for m in TEACHER_MEMBERS}
    auc |= {f"{m} val": val_macro_auroc(yv, p_val[m]) for m in TEACHER_MEMBERS}
    auc["teacher train"] = val_macro_auroc(yt, q)
    auc["teacher val"] = val_macro_auroc(yv, np.mean([p_val[m] for m in TEACHER_MEMBERS], axis=0))
    decision = "go" if auc["teacher train"] < GO_MAX_TRAIN_AUROC else "stop"
    write_json(f"{OUT_DIR}/distill_go_no_go.json", dict(
        protocol_version=PROTOCOL_VERSION, rule=f"go if teacher train macro AUROC < {GO_MAX_TRAIN_AUROC}",
        decision=decision, macro_auroc=auc, split_sha256=SPLIT_SHA256, debug=DEBUG,
        mean_soft_label_on_positives=float(q[yt == 1].mean()), mean_soft_label_on_negatives=float(q[yt == 0].mean())))
    write_json(f"{OUT_DIR}/qc_session_D.json", QC_D)
    print(pd.Series(auc).round(4).to_string())
    print(f"\nGo/no-go: teacher train macro AUROC {auc['teacher train']:.4f} "
          f"(val {auc['teacher val']:.4f}) -> {decision.upper()}")


## 4. Session D2: train the student M10

Same recipe as M08; target = 0.5 x hard label + 0.5 x teacher soft label; model selection on validation macro AUROC against hard labels. QC-D4: the student's val/test exports match M08's.

In [ ]:
# ===================== Session D2: train the student M10 (GPU) =====================
def input_or_output(fname):
    p = f"{OUT_DIR}/{fname}"
    return p if os.path.exists(p) else find_file(fname)

if RUN_STUDENT:
    gng = json.load(open(input_or_output("distill_go_no_go.json")))
    assert gng["debug"] == DEBUG, "go/no-go file comes from a different DEBUG setting"
    if gng["decision"] != "go":
        print(f"Go/no-go = {gng['decision'].upper()} (teacher train macro AUROC {gng['macro_auroc']['teacher train']:.4f}) "
              "- the student is not trained (protocol distillation.go_no_go).")
    else:
        soft = np.load(input_or_output("soft_labels_train.npz"))
        assert (soft["image_index"] == df_train["Image Index"].to_numpy(dtype=str)).all(), "QC-D3: soft labels not in df_train order"
        assert str(soft["split_sha256"]) == SPLIT_SHA256, "QC-D3: soft labels come from another split"
        y_mix = ((1 - KD_LAMBDA) * df_train[CLASSES].to_numpy(dtype=np.float32) + KD_LAMBDA * soft["q"]).astype(np.float32)
        run_stage(STUDENT_CFG, y_train=y_mix, extra_info=dict(distillation=dict(
            teacher="(p_M08 + p_M09) / 2, v2.0.0 Platt calibration", target="(1 - kd_lambda) * y + kd_lambda * q",
            kd_lambda=KD_LAMBDA, teacher_checkpoint_sha256={m: s for m, (_, s) in TEACHER_MEMBERS.items()},
            go_no_go=gng)))

        # QC-D4: student's val/test exports must match M08's images, order, labels and split
        qc_path = input_or_output("qc_session_D.json")
        QC_D = json.load(open(qc_path)) if qc_path else {}
        if DEBUG:
            QC_D["QC-D4"] = "skipped (DEBUG subset)"
        else:
            m08, m10 = find_run_dir(CNN_CFG["model_id"]), find_run_dir(STUDENT_CFG["model_id"])
            bad = []
            for part in ("val", "test"):
                a, b = np.load(f"{m08}/logits_{part}.npz"), np.load(f"{m10}/logits_{part}.npz")
                if not ((a["image_index"] == b["image_index"]).all() and (a["labels"] == b["labels"]).all()
                        and str(a["split_sha256"]) == str(b["split_sha256"])):
                    bad.append(part)
            QC_D["QC-D4"] = "pass" if not bad else f"FAIL ({bad})"
        write_json(f"{OUT_DIR}/qc_session_D.json", QC_D)
        print(QC_D)


## 5. Session D3: evaluation (CPU)

Set `RUN_TEACHER = RUN_STUDENT = False` and `RUN_EVAL = True`. Attach `xrvit-distill-inputs` and the output of the D1+D2 run (it holds the M10 run folder). Calibration and thresholds on validation; test metrics with the same 1000 patient-level resamples as v2.0.0; comparisons Distilled vs CNN / RAD-DINO / Average (Holm over 3); tables T10-T11; outcome in `distill_outcome.json`.

In [ ]:
# ===================== Session D3: evaluation (CPU) =====================
# metric, calibration, threshold and Holm code: copied unchanged from the cascade notebook
# ===================== metrics =====================
HIGHER, LOWER = "higher_is_better", "lower_is_better"
DIRECTION = dict(auroc=HIGHER, auprc=HIGHER, precision=HIGHER, recall=HIGHER, specificity=HIGHER, f1=HIGHER, ece=LOWER,
                 stage2_fraction=LOWER, compute_ratio_vs_raddino=LOWER, labels_missed_by_stage1=LOWER)
THRESHOLD_FREE = {"auroc", "auprc", "ece"}

def auroc_cols(y, s):
    """Mann-Whitney AUROC per class (= sklearn roc_auc_score). No positives or no negatives -> NaN."""
    y = y.astype(bool)
    P = y.sum(0).astype(float); N = len(y) - P
    r = rankdata(s, axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        a = ((r * y).sum(0) - P * (P + 1) / 2) / (P * N)
    a[(P == 0) | (N == 0)] = np.nan
    return a

def ap_cols(y, s):
    """Average precision, step-wise (= sklearn average_precision_score)."""
    out = np.full(y.shape[1], np.nan)
    for c in range(y.shape[1]):
        yc = y[:, c].astype(bool); P = yc.sum()
        if P == 0 or P == len(yc):
            continue
        o = np.argsort(-s[:, c], kind="mergesort")
        ss, yy = s[o, c], yc[o]
        last = np.r_[ss[1:] != ss[:-1], True]
        tp = np.cumsum(yy)[last]; n_pred = np.arange(1, len(yy) + 1)[last]
        out[c] = np.sum(np.diff(np.r_[0.0, tp / P]) * (tp / n_pred))
    return out

def ece_cols(y, p, bins=ECE_BINS):
    out = np.zeros(y.shape[1])
    for c in range(y.shape[1]):
        idx = np.minimum((p[:, c] * bins).astype(int), bins - 1)
        out[c] = sum(abs(p[idx == k, c].mean() - y[idx == k, c].mean()) * (idx == k).mean() for k in range(bins) if (idx == k).any())
    return out

def metric_values(y, score, pred, with_ece=False):
    """{(metric, class): value}; class = label name, MACRO (nanmean over classes) or MICRO."""
    y, pred = y.astype(bool), pred.astype(bool)
    P = y.sum(0); N = len(y) - P
    tp = (pred & y).sum(0); fp = (pred & ~y).sum(0); fn = (~pred & y).sum(0); tn = (~pred & ~y).sum(0)
    pc = dict(auroc=auroc_cols(y, score), auprc=ap_cols(y, score),
              precision=np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0),           # no positive prediction -> 0
              recall=np.where(P > 0, tp / np.maximum(P, 1), np.nan),
              specificity=np.where(N > 0, tn / np.maximum(N, 1), np.nan),
              f1=np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), np.nan))
    if with_ece:
        pc["ece"] = ece_cols(y, score)
    v = {}
    for m, arr in pc.items():
        arr = arr.astype(float)
        v.update({(m, CLASSES[c]): float(arr[c]) for c in range(C)})
        v[(m, "MACRO")] = float(np.nanmean(arr)) if np.isfinite(arr).any() else np.nan
    TP, FP, FN = tp.sum(), fp.sum(), fn.sum()
    v[("precision", "MICRO")] = float(TP / (TP + FP)) if TP + FP else 0.0
    v[("recall", "MICRO")] = float(TP / (TP + FN)) if TP + FN else np.nan
    v[("f1", "MICRO")] = float(2 * TP / (2 * TP + FP + FN)) if 2 * TP + FP + FN else np.nan
    return v

# ---------------- calibration and thresholds (validation only) ----------------
def fit_platt(z, y):
    """p = sigmoid(a_c * logit + b_c), per class, binary NLL, L-BFGS."""
    a, b = np.ones(z.shape[1]), np.zeros(z.shape[1])
    for c in range(z.shape[1]):
        zc, yc = z[:, c].astype(np.float64), y[:, c].astype(np.float64)
        def f(w):
            u = w[0] * zc + w[1]
            g = expit(u) - yc
            return np.mean(np.logaddexp(0, u) - yc * u), np.array([np.mean(g * zc), np.mean(g)])
        a[c], b[c] = minimize(f, [1.0, 0.0], jac=True, method="L-BFGS-B", bounds=[(1e-3, 100), (-50, 50)],
                              options=dict(maxiter=1000)).x
    return a, b

def best_f1_threshold(y, s, tp0=0, fp0=0, fn0=0):
    """Threshold on s maximising F1, given decisions already fixed elsewhere (tp0, fp0, fn0).
    Candidates = unique scores; ties -> smallest threshold. Returns (threshold, f1)."""
    y = y.astype(bool); P = int(y.sum())
    f1_none = 2 * tp0 / max(2 * tp0 + fp0 + fn0 + P, 1)           # option: predict nothing here
    if len(s) == 0:
        return np.inf, f1_none
    order = np.argsort(-s, kind="mergesort")
    s_sorted, y_sorted = s[order], y[order]
    last = np.r_[s_sorted[1:] != s_sorted[:-1], True]
    ctp = np.cumsum(y_sorted)[last]; cfp = np.cumsum(~y_sorted)[last]
    tp, fp, fn = tp0 + ctp, fp0 + cfp, fn0 + P - ctp
    f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    k = int(np.flatnonzero(f1 == f1.max())[-1])
    return (np.inf, f1_none) if f1_none > f1[k] else (float(s_sorted[last][k]), float(f1[k]))

def f1_thresholds(y, s):
    thr, degen = np.zeros(C), np.zeros(C, bool)
    for c in range(C):
        t, f = best_f1_threshold(y[:, c], s[:, c])
        if f == 0 or not np.isfinite(t):
            t, degen[c] = 0.5, True
        thr[c] = t
    return thr, degen

def holm(p):
    p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(np.argsort(p)):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

SYS_MODEL = {"CNN alone": CNN_CFG["model_id"], "RAD-DINO alone": VIT_CFG["model_id"], "Distilled CNN": STUDENT_CFG["model_id"]}
SYSTEM_ORDER = ["CNN alone", "RAD-DINO alone", "Average CNN+RAD-DINO", "Distilled CNN"]
COMPARE = [("auroc", "MACRO"), ("auprc", "MACRO"), ("f1", "MACRO"), ("f1", "MICRO"), ("precision", "MACRO"), ("recall", "MACRO")]
BASELINES = ["CNN alone", "RAD-DINO alone", "Average CNN+RAD-DINO"]          # comparisons: Distilled - each (Holm over 3)

def latency(model_id, bs):
    p = find_file(f"timing_{model_id}.json")
    return json.load(open(p))[f"bs{bs}_median_ms_per_image"] if p else V2_LATENCY_MS[model_id][bs]

if RUN_EVAL:
    # ---- load logits; all systems must be scored on identical images (QC-D4 again, plus split and label order) ----
    NPZ = {}
    for name, mid in SYS_MODEL.items():
        d = find_run_dir(mid)
        assert d, f"no seed-{SEED} logits for {mid} - attach the D2 output (M10) and xrvit-distill-inputs (M08, M09)"
        NPZ[name] = {p: np.load(f"{d}/logits_{p}.npz") for p in ("val", "test")}
        print(f"{name:15s} {d}")
    ref = NPZ["CNN alone"]
    same = all(np.array_equal(ref[p]["image_index"], NPZ[n][p]["image_index"]) and np.array_equal(ref[p]["labels"], NPZ[n][p]["labels"])
               and str(NPZ[n][p]["split_sha256"]) == (str(ref[p]["split_sha256"]) if DEBUG else EXPECTED_SPLIT_SHA256)
               and json.loads(str(NPZ[n][p]["label_order"])) == CLASSES
               for n in NPZ for p in ("val", "test"))
    assert same, "QC-D4: systems differ in images, labels, split hash or label order"
    yv, yt = ref["val"]["labels"].astype(bool), ref["test"]["labels"].astype(bool)
    pid_test = ref["test"]["patient_id"]

    # ---- calibration and thresholds: validation only ----
    PROB, CAL_D = {}, {}
    for name in SYS_MODEL:
        zv, zt = (NPZ[name][p]["logits"].astype(np.float64) for p in ("val", "test"))
        a, b = fit_platt(zv, yv)
        CAL_D[name] = dict(model_id=SYS_MODEL[name], method="platt_per_class", fit_on="val", a=a.tolist(), b=b.tolist())
        PROB[name] = {"val": expit(a * zv + b), "test": expit(a * zt + b)}
    PROB["Average CNN+RAD-DINO"] = {p: (PROB["CNN alone"][p] + PROB["RAD-DINO alone"][p]) / 2 for p in ("val", "test")}
    SYSTEMS = {}
    for name in SYSTEM_ORDER:
        thr, degen = f1_thresholds(yv, PROB[name]["val"])
        SYSTEMS[name] = dict(val=(PROB[name]["val"], PROB[name]["val"] >= thr), test=(PROB[name]["test"], PROB[name]["test"] >= thr),
                             thresholds=thr.tolist(), threshold_degenerate=degen.tolist())
    write_json(f"{OUT_DIR}/calibration_thresholds_distill.json", dict(calibration=CAL_D, thresholds={
        k: dict(strategy="f1_max_per_class", thresholds=v["thresholds"], threshold_degenerate=v["threshold_degenerate"])
        for k, v in SYSTEMS.items()}))
    ms = {bs: {m: latency(m, bs) for m in V2_LATENCY_MS} for bs in (1, 32)}
    r = {bs: ms[bs][CNN_CFG["model_id"]] / ms[bs][VIT_CFG["model_id"]] for bs in (1, 32)}
    COST = {bs: {"CNN alone": r[bs], "RAD-DINO alone": 1.0, "Average CNN+RAD-DINO": 1 + r[bs],
                 "Distilled CNN": r[bs]} for bs in (1, 32)}                  # M10 has M08's architecture

    # ---- patient-level bootstrap (same resamples for every system) ----
    t0 = time.time()
    groups = list(pd.Series(np.arange(len(pid_test))).groupby(pid_test).indices.values())
    keys = list(metric_values(yt, *SYSTEMS[SYSTEM_ORDER[0]]["test"]).keys())
    kpos = {k: i for i, k in enumerate(keys)}
    boot = np.full((N_BOOT, len(SYSTEM_ORDER), len(keys)), np.nan)
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    for b in range(N_BOOT):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        for j, name in enumerate(SYSTEM_ORDER):
            s, p = SYSTEMS[name]["test"]
            v = metric_values(yt[idx], s[idx], p[idx])
            boot[b, j] = [v[k] for k in keys]
    print(f"{N_BOOT} patient-level resamples in {(time.time() - t0) / 60:.1f} min")

    # ---- T10 results (long format like T2; CIs on test only) ----
    rows = []
    for j, name in enumerate(SYSTEM_ORDER):
        for es, y in (("val", yv), ("test", yt)):
            for (m, cls), val in metric_values(y, *SYSTEMS[name][es], with_ece=True).items():
                lo_ = hi_ = np.nan
                if es == "test" and (m, cls) in kpos:
                    col = boot[:, j, kpos[(m, cls)]]; col = col[np.isfinite(col)]
                    if len(col):
                        lo_, hi_ = np.percentile(col, [2.5, 97.5])
                rows.append(dict(system=name, eval_set=es, metric=m, class_name=cls, value=val, ci_low=lo_, ci_high=hi_,
                                 direction=DIRECTION[m], threshold_strategy="none" if m in THRESHOLD_FREE else "f1_max_per_class"))
            rows += [dict(system=name, eval_set=es, metric=f"compute_ratio_vs_raddino_bs{bs}", class_name="GLOBAL", value=COST[bs][name],
                          ci_low=np.nan, ci_high=np.nan, direction=LOWER, threshold_strategy="n/a") for bs in (32, 1)]
    T10 = pd.DataFrame(rows).assign(dataset_id=DATASET_ID, split_id=SPLIT_ID, split_sha256=EXPECTED_SPLIT_SHA256, seed=SEED,
                                    calibration="platt_per_class", n_bootstrap=N_BOOT, protocol_version=PROTOCOL_VERSION)
    save_table(T10, "T10_distill_results")

    # ---- T11 paired comparisons: Distilled - each baseline, Holm over the 3 per metric ----
    q = T10[T10.eval_set == "test"]
    val_of = lambda sname, m, cls: q[(q.system == sname) & (q.metric == m) & (q.class_name == cls)].value.iloc[0]
    comps, jD = [], SYSTEM_ORDER.index("Distilled CNN")
    for m, cls in COMPARE:
        i, fam = kpos[(m, cls)], []
        for base in BASELINES:
            d = boot[:, jD, i] - boot[:, SYSTEM_ORDER.index(base), i]; d = d[np.isfinite(d)]
            delta = val_of("Distilled CNN", m, cls) - val_of(base, m, cls)
            fam.append(dict(metric=m, class_name=cls, system_a="Distilled CNN", system_b=base, delta=delta,
                            ci_low=np.percentile(d, 2.5), ci_high=np.percentile(d, 97.5),
                            delta_rel_pct=100 * delta / val_of(base, m, cls),
                            p_value=max(2 * min((d <= 0).mean(), (d >= 0).mean()), 1 / (N_BOOT + 1))))
        for f, adj in zip(fam, holm([f["p_value"] for f in fam])):
            f["p_holm"] = adj
            f["significant"] = bool((f["ci_low"] > 0 or f["ci_high"] < 0) and adj < ALPHA)
        comps += fam
    T11 = pd.DataFrame(comps)
    save_table(T11, "T11_distill_paired")

    # ---- outcomes (protocol distillation.evaluation.outcomes; primary metric = macro AUROC) ----
    au = T11[(T11.metric == "auroc") & (T11.class_name == "MACRO")].set_index("system_b")
    vs_cnn, vs_rd = au.loc["CNN alone"], au.loc["RAD-DINO alone"]
    OUTCOME = dict(helps=bool(vs_cnn.significant and vs_cnn.delta > 0),
                   matches_raddino=bool(vs_rd.ci_low > -MATCH_MARGIN),
                   beats_raddino=bool(vs_rd.significant and vs_rd.delta > 0))
    OUTCOME["verdict"] = ("beats RAD-DINO" if OUTCOME["beats_raddino"] else
                          "matches RAD-DINO" if OUTCOME["helps"] and OUTCOME["matches_raddino"] else
                          "helps (better than the CNN, below RAD-DINO)" if OUTCOME["helps"] else
                          "fails (no significant gain over the CNN)")
    OUTCOME |= dict(distilled_minus_cnn=[vs_cnn.delta, vs_cnn.ci_low, vs_cnn.ci_high],
                    distilled_minus_raddino=[vs_rd.delta, vs_rd.ci_low, vs_rd.ci_high],
                    compute_ratio_bs32=COST[32]["Distilled CNN"], compute_ratio_bs1=COST[1]["Distilled CNN"])
    write_json(f"{OUT_DIR}/distill_outcome.json", OUTCOME)

    # ---- display ----
    show = [("auroc", "MACRO"), ("auprc", "MACRO"), ("f1", "MACRO"), ("f1", "MICRO"), ("precision", "MACRO"),
            ("recall", "MACRO"), ("specificity", "MACRO"), ("ece", "MACRO")]
    main = pd.DataFrame(index=SYSTEM_ORDER)
    for m, cls in show:
        sub = q[(q.metric == m) & (q.class_name == cls)].set_index("system")
        main[f"{m} {cls.lower()}"] = [(f"{fmt(sub.value[s])} [{fmt(sub.ci_low[s])}, {fmt(sub.ci_high[s])}]"
                                       if np.isfinite(sub.ci_low[s]) else fmt(sub.value[s])) for s in SYSTEM_ORDER]
    main["cost bs32"] = [f"{COST[32][s]:.2f}" for s in SYSTEM_ORDER]
    main["cost bs1"] = [f"{COST[1][s]:.2f}" for s in SYSTEM_ORDER]
    print(f"TEST | {SPLIT_ID} | seed {SEED} (single seed) | no TTA | Platt per class | F1-max thresholds from val | "
          f"{N_BOOT} patient-level resamples")
    display(main)
    display(T11.round(4))
    pc = q[q.class_name.isin(CLASSES) & (q.metric == "auroc")].pivot(index="class_name", columns="system", values="value")[SYSTEM_ORDER]
    pc.insert(0, "support", [int(yt[:, CLASSES.index(c)].sum()) for c in pc.index])
    display(pc.sort_values("support", ascending=False).round(3))
    print(f"\nOUTCOME: {OUTCOME['verdict']} | Distilled - CNN macro AUROC {vs_cnn.delta:+.4f} [{vs_cnn.ci_low:+.4f}, {vs_cnn.ci_high:+.4f}] | "
          f"Distilled - RAD-DINO {vs_rd.delta:+.4f} [{vs_rd.ci_low:+.4f}, {vs_rd.ci_high:+.4f}] | "
          f"cost {COST[32]['Distilled CNN']:.2f}x RAD-DINO (bs32)")
